# Train Siamese BiLSTM (Kaggle T4)

Notebook huan luyen Siamese BiLSTM cho bai toan xep hang passage theo do tuong dong voi cau hoi.

In [1]:
import os
import re
import json
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True

Device: cuda
GPU: Tesla T4


In [2]:
def detect_data_dir():
    candidates = [
        Path('/kaggle/input/datasets/hngphtrn/legals/data/data_ready'),
        Path('/kaggle/input/datasets/hngphtrn/legals/data_ready'),
        Path('/kaggle/input/vnlegal-rag/data/data_ready'),
        Path('/kaggle/input/vnlegal-rag/data_ready'),
        Path('/kaggle/working/vnlegal-rag/data/data_ready'),
        Path('/kaggle/working/legals/data/data_ready'),
        Path('/kaggle/working/data/data_ready'),
        Path('data/data_ready'),
        Path('../data/data_ready'),
    ]
    for p in candidates:
        if p.exists() and (p / 'qa_train.csv').exists() and (p / 'corpus_train.csv').exists():
            return p
    raise FileNotFoundError('Khong tim thay data_ready. Hay kiem tra duong dan dataset tren Kaggle.')

DATA_DIR = detect_data_dir()
ARTIFACT_DIR = Path('/kaggle/working/siamese_lstm_artifacts' if Path('/kaggle').exists() else 'artifacts/siamese_lstm')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_DIR =', DATA_DIR)
print('ARTIFACT_DIR =', ARTIFACT_DIR)

DATA_DIR = /kaggle/input/datasets/hngphtrn/legals/data_ready
ARTIFACT_DIR = /kaggle/working/siamese_lstm_artifacts


In [3]:
qa_train = pd.read_csv(DATA_DIR / 'qa_train.csv', sep='\t')
qa_val = pd.read_csv(DATA_DIR / 'qa_val.csv', sep='\t')
qa_test = pd.read_csv(DATA_DIR / 'qa_test.csv', sep='\t')
corpus_train = pd.read_csv(DATA_DIR / 'corpus_train.csv', sep='\t')
corpus_val = pd.read_csv(DATA_DIR / 'corpus_val.csv', sep='\t')
corpus_test = pd.read_csv(DATA_DIR / 'corpus_test.csv', sep='\t')

for name, df in [('qa_train', qa_train), ('corpus_train', corpus_train)]:
    print(name, df.shape)

required_qa = {'question', 'passage_id', 'macro_domain'}
required_corpus = {'passage_id', 'article_content', 'macro_domain'}
if required_qa - set(qa_train.columns):
    raise ValueError('qa_train thieu cot can thiet')
if required_corpus - set(corpus_train.columns):
    raise ValueError('corpus_train thieu cot can thiet')

qa_train (23311, 14)
corpus_train (7771, 6)


In [4]:
try:
    from underthesea import word_tokenize
    USE_UNDERTHESEA = True
except Exception:
    USE_UNDERTHESEA = False
    print('underthesea chua co. Chay cell sau de cai dat neu can.')

def normalize_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def tokenize(text: str):
    text = normalize_text(text)
    if USE_UNDERTHESEA:
        return [t for t in word_tokenize(text, format='text').split() if t]
    return text.split()

underthesea chua co. Chay cell sau de cai dat neu can.


In [5]:
# Neu can cai underthesea tren Kaggle, bo comment va chay:
# !pip -q install underthesea==6.8.4

In [6]:
PAD = '<PAD>'
UNK = '<UNK>'
MAX_VOCAB = 30000
MAX_Q_LEN = 64
MAX_D_LEN = 256

counter = Counter()
for txt in qa_train['question'].astype(str).tolist():
    counter.update(tokenize(txt))
for txt in corpus_train['article_content'].astype(str).tolist():
    counter.update(tokenize(txt))

vocab_tokens = [PAD, UNK] + [w for w, _ in counter.most_common(MAX_VOCAB - 2)]
stoi = {w: i for i, w in enumerate(vocab_tokens)}
itos = {i: w for w, i in stoi.items()}
print('Vocab size:', len(stoi))

def encode(text, max_len):
    ids = [stoi.get(t, stoi[UNK]) for t in tokenize(text)][:max_len]
    if len(ids) < max_len:
        ids += [stoi[PAD]] * (max_len - len(ids))
    return ids

Vocab size: 25285


In [7]:
corpus_train_map = corpus_train.set_index('passage_id').to_dict(orient='index')
corpus_val_map = corpus_val.set_index('passage_id').to_dict(orient='index')
corpus_test_map = corpus_test.set_index('passage_id').to_dict(orient='index')

domain_to_passages = defaultdict(list)
for _, row in corpus_train.iterrows():
    domain_to_passages[row['macro_domain']].append(row['passage_id'])

In [8]:
class TripletQADataset(Dataset):
    def __init__(self, qa_df, corpus_map, domain_to_passages):
        self.qa_df = qa_df.reset_index(drop=True)
        self.corpus_map = corpus_map
        self.domain_to_passages = domain_to_passages
        self.all_passages = list(corpus_map.keys())

    def __len__(self):
        return len(self.qa_df)

    def sample_negative(self, pos_pid, domain):
        same_domain = self.domain_to_passages.get(domain, [])
        candidates = [p for p in same_domain if p != pos_pid]
        if candidates:
            return random.choice(candidates)
        fallback = [p for p in self.all_passages if p != pos_pid]
        return random.choice(fallback)

    def __getitem__(self, idx):
        row = self.qa_df.iloc[idx]
        q = str(row['question'])
        pos_pid = row['passage_id']
        domain = row['macro_domain']

        if pos_pid not in self.corpus_map:
            neg_pid = self.sample_negative(random.choice(self.all_passages), domain)
            pos_pid = neg_pid

        neg_pid = self.sample_negative(pos_pid, domain)

        pos_text = str(self.corpus_map[pos_pid]['article_content'])
        neg_text = str(self.corpus_map[neg_pid]['article_content'])

        return {
            'anchor': torch.tensor(encode(q, MAX_Q_LEN), dtype=torch.long),
            'positive': torch.tensor(encode(pos_text, MAX_D_LEN), dtype=torch.long),
            'negative': torch.tensor(encode(neg_text, MAX_D_LEN), dtype=torch.long),
        }

In [9]:
train_ds = TripletQADataset(qa_train, corpus_train_map, domain_to_passages)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
print('Train triplets:', len(train_ds))

Train triplets: 23311


In [10]:
class BiLSTMEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=200, hidden_size=256, num_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.pad_idx = pad_idx

    def forward(self, x):
        mask = (x != self.pad_idx).float().unsqueeze(-1)
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        summed = (out * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-6)
        pooled = summed / denom
        return F.normalize(pooled, p=2, dim=1)

In [11]:
class SiameseBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=200, hidden_size=256, num_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()
        self.encoder = BiLSTMEncoder(vocab_size, embed_dim, hidden_size, num_layers, dropout, pad_idx)

    def encode(self, x):
        return self.encoder(x)

    def forward(self, a, p, n):
        return self.encode(a), self.encode(p), self.encode(n)

In [12]:
model = SiameseBiLSTM(vocab_size=len(stoi), pad_idx=stoi[PAD]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)
scaler = GradScaler('cuda', enabled=torch.cuda.is_available())
margin = 0.3

def triplet_loss_cosine(a, p, n, margin=0.3):
    sim_ap = F.cosine_similarity(a, p)
    sim_an = F.cosine_similarity(a, n)
    return F.relu(margin - sim_ap + sim_an).mean()

In [13]:
def train_one_epoch(model, loader):
    model.train()
    total = 0.0
    for batch in tqdm(loader, leave=False):
        a = batch['anchor'].to(device)
        p = batch['positive'].to(device)
        n = batch['negative'].to(device)

        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            za, zp, zn = model(a, p, n)
            loss = triplet_loss_cosine(za, zp, zn, margin=margin)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item() * a.size(0)
    return total / len(loader.dataset)

In [14]:
@torch.no_grad()
def encode_passage_matrix(model, corpus_df, batch_size=64):
    model.eval()
    pids = corpus_df['passage_id'].tolist()
    vecs = []
    for i in range(0, len(pids), batch_size):
        chunk = pids[i:i+batch_size]
        texts = [corpus_df.loc[corpus_df['passage_id'] == pid, 'article_content'].values[0] for pid in chunk]
        x = torch.tensor([encode(t, MAX_D_LEN) for t in texts], dtype=torch.long, device=device)
        z = model.encode(x)
        vecs.append(z.cpu())
    return pids, torch.cat(vecs, dim=0)

@torch.no_grad()
def evaluate_retrieval(model, qa_df, corpus_df, topk=(1,3,5), max_queries=2000):
    model.eval()
    if max_queries is not None and len(qa_df) > max_queries:
        qa_eval = qa_df.sample(max_queries, random_state=42).reset_index(drop=True)
    else:
        qa_eval = qa_df.reset_index(drop=True)

    pids, pmat = encode_passage_matrix(model, corpus_df)
    pid2idx = {p: i for i, p in enumerate(pids)}

    hits = {k: 0 for k in topk}
    rr_sum = 0.0
    valid = 0

    for _, row in qa_eval.iterrows():
        gt = row['passage_id']
        if gt not in pid2idx:
            continue
        q = torch.tensor([encode(row['question'], MAX_Q_LEN)], dtype=torch.long, device=device)
        qv = model.encode(q).cpu()
        sims = torch.matmul(qv, pmat.T).squeeze(0)
        order = torch.argsort(sims, descending=True)
        gt_rank = (order == pid2idx[gt]).nonzero(as_tuple=False)
        if len(gt_rank) == 0:
            continue
        rank = int(gt_rank[0].item()) + 1
        valid += 1
        rr_sum += 1.0 / rank
        for k in topk:
            if rank <= k:
                hits[k] += 1

    metrics = {f'Recall@{k}': (hits[k] / max(valid, 1)) for k in topk}
    metrics['MRR'] = rr_sum / max(valid, 1)
    metrics['n_eval'] = valid
    return metrics

In [15]:
EPOCHS = 10
PATIENCE = 3
best_score = -1.0
wait = 0
best_path = ARTIFACT_DIR / 'siamese_bilstm_best.pt'
history = []

for epoch in range(1, EPOCHS + 1):
    tr_loss = train_one_epoch(model, train_loader)
    val_metrics = evaluate_retrieval(model, qa_val, corpus_val, topk=(1,3,5), max_queries=1500)
    val_score = val_metrics['MRR']
    scheduler.step(val_score)

    row = {'epoch': epoch, 'train_loss': tr_loss, **val_metrics}
    history.append(row)
    print(f"Epoch {epoch:02d} | loss={tr_loss:.4f} | MRR={val_metrics['MRR']:.4f} | R@1={val_metrics['Recall@1']:.4f} | R@5={val_metrics['Recall@5']:.4f}")

    if val_score > best_score:
        best_score = val_score
        wait = 0
        torch.save(model.state_dict(), best_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print('Early stopping triggered.')
            break

print('Best val MRR:', round(best_score, 4))

  0%|          | 0/729 [00:00<?, ?it/s]

Epoch 01 | loss=0.0379 | MRR=0.5294 | R@1=0.4153 | R@5=0.6553


  0%|          | 0/729 [00:00<?, ?it/s]

Epoch 02 | loss=0.0243 | MRR=0.5309 | R@1=0.4127 | R@5=0.6693


  0%|          | 0/729 [00:00<?, ?it/s]

...

Epoch 08 | loss=0.0116 | MRR=0.6149 | R@1=0.5007 | R@5=0.7520


  0%|          | 0/729 [00:00<?, ?it/s]

Epoch 09 | loss=0.0109 | MRR=0.6250 | R@1=0.5087 | R@5=0.7720


  0%|          | 0/729 [00:00<?, ?it/s]

Epoch 10 | loss=0.0101 | MRR=0.6296 | R@1=0.5140 | R@5=0.7727
Best val MRR: 0.6296


In [16]:
model.load_state_dict(torch.load(best_path, map_location=device))
test_metrics = evaluate_retrieval(model, qa_test, corpus_test, topk=(1,3,5,10), max_queries=None)
print('Test metrics:', test_metrics)

Test metrics: {'Recall@1': 0.4944834503510532, 'Recall@3': 0.6746907388833167, 'Recall@5': 0.7432296890672017, 'Recall@10': 0.8234704112337011, 'MRR': 0.6075751937462537, 'n_eval': 2991}


In [17]:
with open(ARTIFACT_DIR / 'tokenizer_vocab.json', 'w', encoding='utf-8') as f:
    json.dump({'stoi': stoi, 'itos': {str(k): v for k, v in itos.items()}}, f, ensure_ascii=False)

with open(ARTIFACT_DIR / 'siamese_meta.json', 'w', encoding='utf-8') as f:
    json.dump({
        'max_q_len': MAX_Q_LEN,
        'max_d_len': MAX_D_LEN,
        'margin': margin,
        'embed_dim': 200,
        'hidden_size': 256,
        'num_layers': 2,
    }, f, ensure_ascii=False, indent=2)

pd.DataFrame(history).to_csv(ARTIFACT_DIR / 'train_history.csv', index=False)
print('Saved artifacts at:', ARTIFACT_DIR)

Saved artifacts at: /kaggle/working/siamese_lstm_artifacts
